# QuickPay Fintech Pipeline
## Part 3: Python Reconciliation Workflow & Part 4: JSON Normalization

**Student Assignment — QuickPay Data Analyst**  
This notebook contains:
- Data quality checks (duplicates, nulls)
- Reconciliation between Ledger and Gateway
- JSON API normalization
- Dashboard source data generation


## Step 1: Import Libraries and Load Data

In [ ]:
import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings('ignore')

# Load raw files
df_ledger = pd.read_csv('../01_data/raw/ledger.csv')
df_gateway = pd.read_csv('../01_data/raw/gateway.csv')
df_clean = pd.read_csv('../01_data/processed/cleaned_transactions.csv')

print("Ledger shape:", df_ledger.shape)
print("Gateway shape:", df_gateway.shape)
print("Cleaned transactions shape:", df_clean.shape)
df_ledger.head()

## Step 2: Data Quality Checks — Nulls and Duplicates

In [ ]:
# Check nulls
print("=== NULL VALUES ===")
print("Ledger nulls:\n", df_ledger.isnull().sum())
print("\nGateway nulls:\n", df_gateway.isnull().sum())

# Check duplicates
print("\n=== DUPLICATES ===")
print("Ledger duplicate transaction IDs:", df_ledger.duplicated(subset='transaction_id').sum())
print("Gateway duplicate transaction IDs:", df_gateway.duplicated(subset='transaction_id').sum())

In [ ]:
# Preview data types
print("Ledger dtypes:\n", df_ledger.dtypes)
print("\nGateway dtypes:\n", df_gateway.dtypes)

## Step 3: Find Records Missing in Gateway

In [ ]:
# Transaction IDs in ledger but NOT in gateway
ledger_ids = set(df_ledger['transaction_id'])
gateway_ids = set(df_gateway['transaction_id'])

missing_in_gateway = df_ledger[~df_ledger['transaction_id'].isin(gateway_ids)].copy()
missing_in_gateway['issue_type'] = 'Missing in Gateway'

print(f"Total ledger records: {len(df_ledger)}")
print(f"Records missing in gateway: {len(missing_in_gateway)}")
missing_in_gateway.head(10)

In [ ]:
# Save to processed folder
missing_in_gateway.to_csv('../01_data/processed/missing_in_gateway.csv', index=False)
print("Saved: missing_in_gateway.csv")

## Step 4: Find Records Missing in Ledger

In [ ]:
# Transaction IDs in gateway but NOT in ledger
missing_in_ledger = df_gateway[~df_gateway['transaction_id'].isin(ledger_ids)].copy()
missing_in_ledger['issue_type'] = 'Missing in Ledger'

print(f"Total gateway records: {len(df_gateway)}")
print(f"Records missing in ledger: {len(missing_in_ledger)}")
missing_in_ledger.head(10)

In [ ]:
missing_in_ledger.to_csv('../01_data/processed/missing_in_ledger.csv', index=False)
print("Saved: missing_in_ledger.csv")

## Step 5: Identify Amount Mismatches

In [ ]:
# Merge common records (inner join on transaction_id)
df_merged = df_ledger.merge(df_gateway, on='transaction_id', suffixes=('_ledger', '_gateway'))
print(f"Common records (matched in both): {len(df_merged)}")

# Amount mismatch = absolute difference > $1
df_merged['amount_diff'] = (df_merged['amount_ledger'] - df_merged['amount_gateway']).abs()
amount_mismatches = df_merged[df_merged['amount_diff'] > 1.0].copy()
amount_mismatches['issue_type'] = 'Amount Mismatch'

print(f"Amount mismatches found: {len(amount_mismatches)}")
amount_mismatches[['transaction_id','amount_ledger','amount_gateway','amount_diff','issue_type']].head(10)

In [ ]:
amount_mismatches.to_csv('../01_data/processed/amount_mismatches.csv', index=False)
print("Saved: amount_mismatches.csv")

## Step 6: Identify Status Mismatches

In [ ]:
# Status mismatch = ledger and gateway report different statuses for same transaction
status_mismatches = df_merged[df_merged['status_ledger'] != df_merged['status_gateway']].copy()
status_mismatches['issue_type'] = 'Status Mismatch'

print(f"Status mismatches found: {len(status_mismatches)}")
status_mismatches[['transaction_id','status_ledger','status_gateway','issue_type']].head(10)

In [ ]:
status_mismatches.to_csv('../01_data/processed/status_mismatches.csv', index=False)
print("Saved: status_mismatches.csv")

## Step 7: Build Final Reconciliation Report

In [ ]:
# Combine all issues into one report
recon_rows = []

for _, row in missing_in_gateway.iterrows():
    recon_rows.append({
        'transaction_id': row['transaction_id'],
        'issue_type': 'Missing in Gateway',
        'ledger_amount': row['amount'],
        'gateway_amount': None,
        'ledger_status': row['status'],
        'gateway_status': None,
        'merchant': row['merchant_name']
    })

for _, row in missing_in_ledger.iterrows():
    recon_rows.append({
        'transaction_id': row['transaction_id'],
        'issue_type': 'Missing in Ledger',
        'ledger_amount': None,
        'gateway_amount': row['amount'],
        'ledger_status': None,
        'gateway_status': row['status'],
        'merchant': 'Unknown'
    })

for _, row in amount_mismatches.iterrows():
    recon_rows.append({
        'transaction_id': row['transaction_id'],
        'issue_type': 'Amount Mismatch',
        'ledger_amount': row['amount_ledger'],
        'gateway_amount': row['amount_gateway'],
        'ledger_status': row['status_ledger'],
        'gateway_status': row['status_gateway'],
        'merchant': row['merchant_name']
    })

# Add status mismatches not already covered
covered_ids = {r['transaction_id'] for r in recon_rows}
for _, row in status_mismatches.iterrows():
    if row['transaction_id'] not in covered_ids:
        recon_rows.append({
            'transaction_id': row['transaction_id'],
            'issue_type': 'Status Mismatch',
            'ledger_amount': row['amount_ledger'],
            'gateway_amount': row['amount_gateway'],
            'ledger_status': row['status_ledger'],
            'gateway_status': row['status_gateway'],
            'merchant': row['merchant_name']
        })

df_recon = pd.DataFrame(recon_rows)
df_recon.to_csv('../01_data/processed/reconciliation_report.csv', index=False)

print(f"Total reconciliation issues: {len(df_recon)}")
print("\nIssue breakdown:")
print(df_recon['issue_type'].value_counts())
df_recon.head(10)

## Step 8: Generate Summary Metrics

In [ ]:
# Calculate amount at risk
amount_at_risk = (
    missing_in_gateway['amount'].sum() +
    missing_in_ledger['amount'].sum() +
    amount_mismatches['amount_diff'].sum()
)

summary_metrics = {
    "total_ledger_rows": int(len(df_ledger)),
    "total_gateway_rows": int(len(df_gateway)),
    "missing_in_gateway_count": int(len(missing_in_gateway)),
    "missing_in_ledger_count": int(len(missing_in_ledger)),
    "amount_mismatch_count": int(len(amount_mismatches)),
    "status_mismatch_count": int(len(status_mismatches)),
    "reconciliation_issue_count": int(len(df_recon)),
    "ledger_total_amount": round(float(df_ledger['amount'].sum()), 2),
    "gateway_total_amount": round(float(df_gateway['amount'].sum()), 2),
    "amount_at_risk": round(float(amount_at_risk), 2)
}

with open('summary_metrics.json', 'w') as f:
    json.dump(summary_metrics, f, indent=2)

print("Summary Metrics:")
for k, v in summary_metrics.items():
    print(f"  {k}: {v}")

---
## Part 4: JSON API Normalization

In [ ]:
# Load the nested JSON file
with open('../01_data/raw/api_response_sample.json') as f:
    api_data = json.load(f)

print("Top-level keys:", list(api_data.keys()))
print("Data keys:", list(api_data['data'].keys()))
print("Sample transaction keys:", list(api_data['data']['transactions'][0].keys()))

In [ ]:
# Flatten nested JSON into tabular format
transactions = api_data['data']['transactions']

rows = []
for tx in transactions:
    rows.append({
        'transaction_id': tx['id'],
        'timestamp': tx['timestamp'],
        'merchant_id': tx['merchant']['id'],
        'merchant_name': tx['merchant']['name'],
        'merchant_region': tx['merchant']['region'],
        'payment_amount': tx['payment']['amount'],
        'payment_currency': tx['payment']['currency'],
        'payment_method': tx['payment']['method'],
        'user_id': tx['user']['id'],
        'user_country': tx['user']['country'],
        'status': tx['status'],
        'risk_score': tx['risk']['score'],
        'risk_flags': ','.join(tx['risk']['flags']) if tx['risk']['flags'] else ''
    })

df_api = pd.DataFrame(rows)
print(f"Flattened API data: {df_api.shape[0]} rows, {df_api.shape[1]} columns")
df_api.head()

In [ ]:
# Clean column names and convert date/time
df_api.columns = [c.lower().replace(' ', '_') for c in df_api.columns]
df_api['timestamp'] = pd.to_datetime(df_api['timestamp'])
df_api['date'] = df_api['timestamp'].dt.date
df_api['hour'] = df_api['timestamp'].dt.hour

print("Columns:", df_api.columns.tolist())
print("dtypes:\n", df_api.dtypes)
df_api.head()

In [ ]:
df_api.to_csv('../01_data/processed/api_normalized.csv', index=False)
print("Saved: api_normalized.csv")
print(f"Shape: {df_api.shape}")

## Dashboard Source Data Generation

In [ ]:
# Daily summary
df_daily = df_clean.groupby('transaction_date').agg(
    total_transactions=('transaction_id', 'count'),
    total_gmv_usd=('amount_usd', 'sum'),
    success_count=('status', lambda x: (x == 'captured').sum()),
    chargeback_count=('status', lambda x: (x == 'chargeback').sum()),
    avg_risk_score=('risk_score', 'mean')
).reset_index()
df_daily['success_rate_pct'] = (df_daily['success_count'] / df_daily['total_transactions'] * 100).round(2)
df_daily['total_gmv_usd'] = df_daily['total_gmv_usd'].round(2)
df_daily.to_csv('../01_data/processed/daily_summary.csv', index=False)
print("Daily summary shape:", df_daily.shape)
df_daily.head()

In [ ]:
# Payment method breakdown
df_pm = df_clean.groupby('payment_method').agg(
    transaction_count=('transaction_id', 'count'),
    total_gmv_usd=('amount_usd', 'sum'),
    avg_amount=('amount_usd', 'mean')
).reset_index().round(2)
df_pm.to_csv('../01_data/processed/payment_method_breakdown.csv', index=False)

# Region breakdown
df_region = df_clean.groupby('region').agg(
    transaction_count=('transaction_id', 'count'),
    total_gmv_usd=('amount_usd', 'sum'),
    avg_risk_score=('risk_score', 'mean'),
    high_value_count=('high_value_flag', 'sum'),
    high_risk_count=('high_risk_flag', 'sum')
).reset_index().round(2)
df_region.to_csv('../01_data/processed/region_breakdown.csv', index=False)

# Merchant performance summary
df_mperf = df_clean.groupby('merchant_name').agg(
    total_transactions=('transaction_id', 'count'),
    total_gmv_usd=('amount_usd', 'sum'),
    chargeback_count=('status', lambda x: (x == 'chargeback').sum()),
    avg_risk_score=('risk_score', 'mean'),
    high_value_count=('high_value_flag', 'sum')
).reset_index()
df_mperf['chargeback_ratio_pct'] = (df_mperf['chargeback_count'] / df_mperf['total_transactions'] * 100).round(2)
df_mperf['total_gmv_usd'] = df_mperf['total_gmv_usd'].round(2)
df_mperf.to_csv('../01_data/processed/merchant_performance_summary.csv', index=False)

print("Payment method breakdown:", df_pm.shape)
print("Region breakdown:", df_region.shape)
print("Merchant performance:", df_mperf.shape)
print("\nAll dashboard source files saved!")